In [ ]:
# Cell 1: Load model and imports

import os
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

MODEL_ID = 'Banjomenny/DisInfoBERT'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.to(device)
model.eval()

print(f'Model loaded from {MODEL_ID}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Section 1 — Model & Dataset: Baseline Evaluation
# Reproduce A2 test split, evaluate fine-tuned RoBERTa, classification report + confusion matrix

import pandas as pd, re, os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

# ── Load raw datasets (same sources as A2) ───────────────
DATA = './datasets'
info_op    = pd.read_csv(os.path.join(DATA, 'john123462546_english-ioa-tweets', 'ioa_tweets_en.csv'), low_memory=False)
legitimate = pd.read_csv(os.path.join(DATA, 'kaushiksuresh147_political-tweets', 'Political_tweets.csv'), low_memory=False)
general    = pd.read_csv(os.path.join(DATA, 'train.csv', 'train.csv'), low_memory=False)

legitimate.columns = legitimate.columns.str.strip()
info_op.columns    = info_op.columns.str.strip()
info_op = info_op.rename(columns={'tweet_text': 'text'})

legitimate['Label'] = 0
info_op['Label']    = 1

general_clean = general[['tweet']].copy()
general_clean = general_clean.rename(columns={'tweet': 'text'})
general_clean['Label'] = 0
general_clean['is_retweet'] = False

GENERAL_SAMPLE_SIZE = min(len(legitimate), len(general_clean))
general_sample = general_clean.sample(n=GENERAL_SAMPLE_SIZE, random_state=42)

keep_cols = ['text', 'is_retweet', 'Label']
df = pd.concat([info_op[keep_cols], legitimate[keep_cols], general_sample[keep_cols]], ignore_index=True)

# ── Same cleaning as A2 ─────────────────────────────────
df_clean = df.drop_duplicates(subset='text')
df_clean = df_clean[df_clean['is_retweet'] == False]
df_clean = df_clean.dropna(subset=['text'])
df_clean = df_clean[df_clean['text'].str.strip() != '']

def clean_text(text):
    text = str(text)
    text = re.sub(r'^RT\s+@?\w*:?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^RR\s+RT\s+@?\w*:?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'\bamp\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['text'] = df_clean['text'].apply(clean_text)
df_clean = df_clean[df_clean['text'].str.len() > 20]
df_clean = df_clean.drop(columns='is_retweet')

X = df_clean['text']
y = df_clean['Label']
_, X_test_text, _, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_test_text = X_test_text.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f'Test set reproduced: {len(X_test_text):,} rows')
print(f'  Organic: {(y_test == 0).sum():,}  |  IO: {(y_test == 1).sum():,}')

# ── Batch inference on full test set ─────────────────────
print('\nRunning inference on test set...')
BATCH_SIZE = 64
all_preds = []

for i in range(0, len(X_test_text), BATCH_SIZE):
    batch_texts = X_test_text[i:i+BATCH_SIZE].tolist()
    inputs = tokenizer(
        batch_texts, truncation=True, padding='max_length',
        max_length=128, return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
    all_preds.extend(preds)

    if (i // BATCH_SIZE) % 50 == 0:
        print(f'  Batch {i // BATCH_SIZE + 1}/{(len(X_test_text) + BATCH_SIZE - 1) // BATCH_SIZE}')

y_pred = np.array(all_preds)

# ── Classification Report ────────────────────────────────
print()
print('=' * 55)
print('  SECTION 1 — CLASSIFICATION REPORT')
print('  Fine-tuned Twitter-RoBERTa on IO Detection')
print('=' * 55)
print()
print(classification_report(
    y_test, y_pred,
    target_names=['Organic', 'IO'],
    digits=4
))

roberta_f1  = f1_score(y_test, y_pred, average='weighted')
roberta_acc = accuracy_score(y_test, y_pred)
print(f'  Weighted F1:  {roberta_f1:.4f}')
print(f'  Accuracy:     {roberta_acc:.4f}')

# ── Confusion Matrix ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fine-tuned Twitter-RoBERTa — Confusion Matrix', fontsize=14)

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Organic', 'IO']).plot(
    ax=axes[0], cmap='Blues', values_format=','
)
axes[0].set_title('Raw Counts')

cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
ConfusionMatrixDisplay(cm_norm, display_labels=['Organic', 'IO']).plot(
    ax=axes[1], cmap='Blues', values_format='.3f'
)
axes[1].set_title('Normalized')

plt.tight_layout()
plt.savefig('section1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: section1_confusion_matrix.png')

In [ ]:
# Section 2 — Fine-Tuning Results
# Hyperparameters used, text preprocessing kept/dropped, before vs after comparison

# ── Training hyperparameters ─────────────────────────────
print('=' * 60)
print('  SECTION 2 — FINE-TUNING & IMPROVEMENTS')
print('=' * 60)

print('\n  Base model:  cardiffnlp/twitter-roberta-base (125M params)')
print('  Task:        Binary classification (Organic=0, IO=1)')
print('  Tokenizer:   RoBERTa BPE, max_length=128')

print('\n  Training hyperparameters:')
print('  ┌─────────────────────────┬────────────┐')
print('  │ Hyperparameter          │ Value      │')
print('  ├─────────────────────────┼────────────┤')
print('  │ Learning rate           │ 2e-5       │')
print('  │ Batch size (train)      │ 16         │')
print('  │ Batch size (eval)       │ 32         │')
print('  │ Epochs                  │ 3          │')
print('  │ Optimizer               │ AdamW      │')
print('  │ FP16 mixed precision    │ Yes        │')
print('  │ Metric for best model   │ F1 (wt.)   │')
print('  │ Evaluation strategy     │ per epoch  │')
print('  │ Seed                    │ 42         │')
print('  └─────────────────────────┴────────────┘')

# ── Text preprocessing kept vs dropped ───────────────────
print('\n  Text preprocessing (NLP equivalent of feature engineering):')
print('  ┌──────────────────────────────┬──────────┬─────────────────────────────┐')
print('  │ Preprocessing Step           │ Kept?    │ Reason                      │')
print('  ├──────────────────────────────┼──────────┼─────────────────────────────┤')
print('  │ URL removal                  │ Removed  │ URLs are noise, not content │')
print('  │ @mention removal             │ Removed  │ User handles leak identity  │')
print('  │ Hashtag removal              │ Removed  │ Hashtags vary by campaign   │')
print('  │ Retweet prefix strip         │ Removed  │ RT markers not predictive   │')
print('  │ Non-ASCII removal            │ Removed  │ Emoji/special chars = noise │')
print('  │ Short text filter (>20 char) │ Applied  │ Too-short posts lack signal │')
print('  │ Lowercasing                  │ Kept     │ RoBERTa tokenizer handles   │')
print('  │ Punctuation                  │ Kept     │ RoBERTa uses subword tokens │')
print('  │ Stop words                   │ Kept     │ Contextual model uses them  │')
print('  └──────────────────────────────┴──────────┴─────────────────────────────┘')

# ── Before vs After comparison ───────────────────────────
# A1 baseline: TF-IDF + Logistic Regression (from context doc)
baseline_f1  = 0.83
baseline_acc = 0.83

f1_change  = roberta_f1 - baseline_f1
acc_change = roberta_acc - baseline_acc

print('\n  Before vs After comparison:')
print('  ┌──────────────────┬─────────────────────┬─────────────────────┬──────────┐')
print('  │ Metric           │ Baseline (TF-IDF+LR)│ Fine-tuned RoBERTa  │ Change   │')
print('  ├──────────────────┼─────────────────────┼─────────────────────┼──────────┤')
print(f'  │ F1 (weighted)    │ {baseline_f1:.4f}              │ {roberta_f1:.4f}              │ {f1_change:+.4f}  │')
print(f'  │ Accuracy         │ {baseline_acc:.4f}              │ {roberta_acc:.4f}              │ {acc_change:+.4f}  │')
print(f'  │ Model type       │ Bag-of-words         │ Contextual (attention)│ upgrade  │')
print(f'  │ Parameters       │ TF-IDF vocab (~50K)  │ 125M                 │ larger   │')
print('  └──────────────────┴─────────────────────┴─────────────────────┴──────────┘')

print(f'\n  Summary: RoBERTa {"improves" if f1_change > 0 else "underperforms"} over TF-IDF+LR baseline')
print(f'  by {abs(f1_change):.4f} F1 points. RoBERTa captures word order and context')
print(f'  that bag-of-words TF-IDF misses, which matters for detecting')
print(f'  coordinated IO content that relies on framing rather than keywords.')

In [ ]:
# Section 3 — Prediction Pipeline Demo
# 3 test-set examples: 1 Organic + 2 IO, with real label, predicted label, confidence

def predict_post(text, real_label):
    """Run a single tweet through the fine-tuned RoBERTa model."""
    label_map = {0: 'Organic', 1: 'IO'}
    inputs = tokenizer(
        text, truncation=True, padding='max_length',
        max_length=128, return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    pred_idx   = torch.argmax(probs).item()
    pred_name  = label_map[pred_idx]
    real_name  = label_map[real_label]
    confidence = probs[pred_idx].item()
    correct    = pred_idx == real_label

    print('=' * 60)
    print(f'  Text:       {text[:80]}{"..." if len(text) > 80 else ""}')
    print(f'  Real label: {real_name}')
    print(f'  Predicted:  {pred_name}  {"CORRECT" if correct else "INCORRECT"}')
    print(f'  Confidence: {confidence:.4f}')
    print(f'  P(Organic): {probs[0]:.4f}   P(IO): {probs[1]:.4f}')
    print('=' * 60)
    return pred_name, real_name, confidence, correct

# ── Pick 1 Organic + 2 IO from the test set ──────────────
organic_idx = y_test[y_test == 0].sample(1, random_state=7).index
io_idx      = y_test[y_test == 1].sample(2, random_state=7).index

print('=' * 60)
print('  SECTION 3 — PREDICTION PIPELINE DEMO')
print('  3 real test-set examples through fine-tuned RoBERTa')
print('=' * 60)

results = []

print('\nExample 1 — Organic tweet')
r1 = predict_post(X_test_text[organic_idx[0]], y_test[organic_idx[0]])
results.append(r1)

print('\nExample 2 — IO tweet')
r2 = predict_post(X_test_text[io_idx[0]], y_test[io_idx[0]])
results.append(r2)

print('\nExample 3 — IO tweet')
r3 = predict_post(X_test_text[io_idx[1]], y_test[io_idx[1]])
results.append(r3)

# ── Summary table ────────────────────────────────────────
print('\n  Summary:')
print('  ┌─────────┬──────────┬───────────┬────────────┬─────────┐')
print('  │ Example │ Real     │ Predicted │ Confidence │ Correct │')
print('  ├─────────┼──────────┼───────────┼────────────┼─────────┤')
for i, (pred, real, conf, ok) in enumerate(results, 1):
    print(f'  │ {i}       │ {real:<8} │ {pred:<9} │ {conf:.4f}     │ {"Yes" if ok else "No":>7} │')
print('  └─────────┴──────────┴───────────┴────────────┴─────────┘')

n_correct = sum(1 for _, _, _, ok in results if ok)
print(f'\n  {n_correct}/3 predictions correct.')
if n_correct < 3:
    print('  Note: Misclassifications may indicate the model struggles with')
    print('  tweets that lack strong stylistic IO signals or organic tweets')
    print('  that happen to discuss political topics similar to IO content.')